In [ ]:
# 1. Instala a biblioteca pyarrow, necessária para o Pandas ler Parquet
# O '!' no início executa como um comando no terminal do Colab
!pip install pyarrow

# 2. Importa a biblioteca Pandas
import pandas as pd


In [ ]:
# 3. Define o nome do arquivo (exatamente como você fez o upload)
# ATENÇÃO: Se o nome do arquivo for 'consulta_baco.parquet' (como no seu script original),
# ajuste o nome aqui. Usarei 'consulta_banco.parquet' como você pediu.
nome_arquivo = 'dataset_mock.csv'

# 4. Tenta carregar o arquivo Parquet para um DataFrame
try:
    # O 'engine='pyarrow'' garante que estamos usando a biblioteca que acabamos de instalar
    df = pd.read_csv(nome_arquivo, engine='pyarrow')
    df['data_da_viagem'] = pd.to_datetime(df['data_da_viagem'])
    print(f"Arquivo '{nome_arquivo}' carregado com sucesso!")
    print("-" * 30)

    # --- INÍCIO DO TRATAMENTO E ANÁLISE ---

    # 5. Visualiza as primeiras linhas do DataFrame
    print("\n--- 5 Primeiras Linhas (df.head()) ---")
    print(df.head())

    # 6. Verifica os tipos de dados e a contagem de nulos
    # Muito importante para o tratamento!
    print("\n--- Informações do DataFrame (df.info()) ---")
    df.info()

except FileNotFoundError:
    print(f"\nERRO: Arquivo '{nome_arquivo}' não encontrado.")
    print("Por favor, verifique se o upload foi concluído e se o nome do arquivo está correto.")
except Exception as e:
    print(f"\nOcorreu um erro inesperado ao ler o arquivo: {e}")

In [ ]:
# Exemplo para converter colunas de data
df['data_viagem'] = pd.to_datetime(df['data_viagem'])
df['data_ultima_venda'] = pd.to_datetime(df['data_ultima_venda'])
df['data_primeira_venda'] = pd.to_datetime(df['data_primeira_venda'])

In [ ]:
# Gera estatísticas como média, mediana, min, max, etc.
print(df.describe())

In [ ]:
# --- CÉLULA DE DIAGNÓSTICO: VERIFICAR COLUNAS ---
# Isso vai nos mostrar exatamente quais colunas estão no arquivo Parquet

if 'df' in locals():
    print("--- Colunas Atuais no DataFrame 'df' ---")
    print(list(df.columns))
else:
    print("ERRO: O DataFrame 'df' não foi carregado. Rode a Célula 2 primeiro.")

In [ ]:
# --- CÉLULA 5: ANÁLISE DE CORRIDAS (Corrigida) ---

# 1. Quantos boletos (linhas) cada CORRIDA_ID tem?
# CORREÇÃO: Usando 'corrida_id' (minúsculo)
corridas_populares = df['corrida_id'].value_counts()

print("--- Corridas Mais Populares (com mais vendas) ---")
print(corridas_populares.head(10))

# 2. Quantas corridas únicas você tem no total?
print(f"\nTotal de corridas únicas no dataset: {len(corridas_populares)}")

In [ ]:
# --- CÉLULA 6: FILTRAGEM AUTOMÁTICA (Corrigida) ---

# 1. Verifica se a variável 'corridas_populares' existe (criada na Célula 5)
if 'corridas_populares' not in locals() or corridas_populares.empty:
    print("ERRO: A variável 'corridas_populares' não foi encontrada ou está vazia.")
    print("Por favor, execute a Célula 5 (Análise de Corridas) primeiro.")
else:
    # 2. Pega o ID mais popular (o primeiro item do índice da Series)
    ID_ALVO = corridas_populares.index[0]

    # 3. Pega o número de vendas (o primeiro valor da Series) para o print
    num_vendas = corridas_populares.iloc[0]

    print(f"--- Foco automático na Corrida ID: {ID_ALVO} (Com {num_vendas} vendas) ---")

    # 4. Criamos um novo DataFrame contendo dados APENAS dessa corrida
    # CORREÇÃO: Usando 'corrida_id' (minúsculo)
    df_filtrado = df[df['corrida_id'] == ID_ALVO].copy()

    print(f"Total de boletos encontrados para esta corrida: {len(df_filtrado)}")

In [ ]:
# --- CÉLULA 7: AGREGAÇÃO DIÁRIA (DA CORRIDA FILTRADA) ---
import numpy as np

# --- 1. AGREGAR OS DADOS POR DIA (DA CORRIDA FILTRADA) ---
# (Note que estamos usando 'df_filtrado')
df_diario = df_filtrado.groupby('data_viagem').size().reset_index(name='total_passagens_dia')

# --- 2. CRIAR UM ÍNDICE DE TEMPO COMPLETO ---
df_diario = df_diario.set_index('data_viagem')
df_diario = df_diario.asfreq('D')

# --- 3. PREENCHER DIAS SEM VENDA COM ZERO ---
df_diario['total_passagens_dia'] = df_diario['total_passagens_dia'].fillna(0)
df_diario = df_diario.reset_index()

print("--- DataFrame Diário Agregado (df_diario) ---")
print(f"Série criada com {len(df_diario)} dias de dados.")
print(df_diario.head())

In [ ]:
# --- CÉLULA 8 (NOVA): FEATURES DE DATA (Dia da Semana/Ano) ---

# O .dt.dayofweek retorna um número (0 = Segunda, 1 = Terça, ..., 6 = Domingo)
df_diario['dia_da_semana'] = df_diario['data_viagem'].dt.dayofweek

# O dia do ano (1 a 365) é essencial para a sazonalidade anual
df_diario['dia_do_ano'] = df_diario['data_viagem'].dt.dayofyear

print("--- DataFrame com Dia da Semana e Dia do Ano ---")
print(df_diario.head())

In [ ]:
# --- 4. SENO E COSSENO DA DATA DE VIAGEM (SAZONALIDADE) ---

periodo_ano = 365.25
periodo_semana = 7

# --- Sazonalidade Anual (baseada no dia do ano) ---
df_diario['dia_ano_sin'] = np.sin(2 * np.pi * df_diario['dia_do_ano'] / periodo_ano)
df_diario['dia_ano_cos'] = np.cos(2 * np.pi * df_diario['dia_do_ano'] / periodo_ano)

# --- Sazonalidade Semanal (baseada no dia da semana) ---
df_diario['dia_semana_sin'] = np.sin(2 * np.pi * df_diario['dia_da_semana'] / periodo_semana)
df_diario['dia_semana_cos'] = np.cos(2 * np.pi * df_diario['dia_da_semana'] / periodo_semana)


print("--- DataFrame com Features Cíclicas (Seno/Cosseno) ---")
# Mostrando apenas as colunas relevantes para a verificação
print(df_diario[['data_viagem', 'dia_ano_sin', 'dia_ano_cos', 'dia_semana_sin', 'dia_semana_cos']].head())

In [ ]:
# --- 5. COLUNA PARA VERIFICAR SE É FERIADO ---

# Primeiro, instala a biblioteca
!pip install holidays

import holidays

# Pega o range de anos dos nossos dados
anos = df_diario['data_viagem'].dt.year.unique()

# Cria o "calendário" de feriados do Brasil, especificando o estado de Roraima (RR)
# Isso inclui feriados nacionais + estaduais de RR + municipais de Boa Vista
# Para Boa Vista, usamos o sub-market 'RR-Boa Vista'
feriados_br_rr = holidays.BR(state='RR', years=anos)
# (Opcional, se quiser incluir os da capital)
# feriados_bvb = holidays.BR(subdiv='RR-Boa Vista', years=anos)
# feriados_totais = feriados_br_rr + feriados_bvb

# Cria a coluna: 1 se a data ESTIVER no calendário de feriados, 0 se NÃO ESTIVER
df_diario['eh_feriado'] = df_diario['data_viagem'].isin(feriados_br_rr).astype(int)

print("--- DataFrame com Feriados ---")
print(f"Total de feriados encontrados no período: {df_diario['eh_feriado'].sum()}")
# Mostra os primeiros 5 feriados encontrados nos dados
print(df_diario[df_diario['eh_feriado'] == 1].head())

In [ ]:
# --- 6. "LAG" FEATURES ---

# Garante a ordenação pela data (crucial para o shift funcionar)
df_diario = df_diario.sort_values('data_viagem')

# Cria lag de 1 dia (vendas de ontem)
df_diario['lag_1_dia'] = df_diario['total_passagens_dia'].shift(1)

# Cria lag de 7 dias (vendas do mesmo dia na semana passada)
df_diario['lag_7_dias'] = df_diario['total_passagens_dia'].shift(7)

# Cria lag de 365 dias (vendas do mesmo dia no ano passado)
# Isso SÓ FUNCIONA porque você buscou 24 meses (2 anos) de dados
df_diario['lag_365_dias'] = df_diario['total_passagens_dia'].shift(365)

# Os primeiros 365 dias ficarão com 'NaN' (nulo) no lag_365_dias
print("--- DataFrame com Features de Lag ---")
# Vamos olhar os dados após 1 ano, para ver o lag_365 funcionando
print(df_diario.tail())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# --- 7. DEFINIÇÃO DO MODELO E PREDIÇÃO ---

# 1. Limpar os dados: Remove os NaNs criados pelos Lags
# (Isso vai remover os primeiros 365 dias, que não têm 'lag_365_dias')
df_modelo = df_diario.dropna()

print(f"\nUsando {len(df_modelo)} dias para treino/teste (após remover NaNs dos lags).")

# 2. Definir Features (X) e Alvo (y)
features = [
    'dia_da_semana',
    'eh_feriado',
    'dia_ano_sin',
    'dia_ano_cos',
    'dia_semana_sin',
    'dia_semana_cos',
    'lag_1_dia',
    'lag_7_dias',
    'lag_365_dias' # A feature mais forte!
]
target = 'total_passagens_dia'

X = df_modelo[features]
y = df_modelo[target]

# 3. Dividir em Treino e Teste (IMPORTANTE PARA SÉRIE TEMPORAL)
# NUNCA embaralhe dados de tempo (shuffle=False)
# Vamos usar os últimos 20% dos dados (aprox. 3 meses) para teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 4. Treinar o Modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# 5. Fazer Previsões
predicoes = modelo.predict(X_test)
# Arredonda e garante que não haja previsões negativas
predicoes[predicoes < 0] = 0
predicoes = np.round(predicoes)

# 6. Avaliar o Modelo
rmse = np.sqrt(mean_squared_error(y_test, predicoes))
print(f"\n--- Avaliação do Modelo ---")
print(f"RMSE (Erro Médio de Passagens): {rmse:.2f}")

# 7. Plotar resultados
plt.figure(figsize=(15, 6))
plt.title(f'Previsão vs. Real (RMSE: {rmse:.2f})')
# Usamos o índice do y_test para plotar corretamente no tempo
plt.plot(y_test.index, y_test, label='Valores Reais', alpha=0.7)
plt.plot(y_test.index, predicoes, label='Previsões do Modelo', linestyle='--')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

print("--- Treinando Modelo 2: Random Forest Regressor ---")

# 1. Criar o Modelo
# n_estimators=100 significa que ele vai criar 100 árvores
# random_state=42 garante que você terá o mesmo resultado se rodar de novo
# n_jobs=-1 usa todos os núcleos do seu processador para treinar mais rápido
rf_modelo = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Treinar o Modelo
rf_modelo.fit(X_train, y_train)

# 3. Fazer Previsões
rf_predicoes = rf_modelo.predict(X_test)
# Arredonda e garante que não haja previsões negativas
rf_predicoes[rf_predicoes < 0] = 0
rf_predicoes = np.round(rf_predicoes)

# 4. Avaliar o Modelo
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predicoes))
print(f"RMSE (Random Forest): {rf_rmse:.2f}")

# 5. Plotar resultados
plt.figure(figsize=(15, 6))
plt.title(f'Previsão (Random Forest) vs. Real (RMSE: {rf_rmse:.2f})')
plt.plot(y_test.index, y_test, label='Valores Reais', alpha=0.7)
plt.plot(y_test.index, rf_predicoes, label='Previsões (RF)', linestyle='--')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 1. Instalar a biblioteca
!pip install lightgbm

import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

print("--- Treinando Modelo 3: LightGBM ---")

# 1. Criar o Modelo
lgb_modelo = lgb.LGBMRegressor(random_state=42, n_jobs=-1)

# 2. Treinar o Modelo
lgb_modelo.fit(X_train, y_train)

# 3. Fazer Previsões
lgb_predicoes = lgb_modelo.predict(X_test)
# Arredonda e garante que não haja previsões negativas
lgb_predicoes[lgb_predicoes < 0] = 0
lgb_predicoes = np.round(lgb_predicoes)

# 4. Avaliar o Modelo
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_predicoes))
print(f"RMSE (LightGBM): {lgb_rmse:.2f}")

# 5. Plotar resultados
plt.figure(figsize=(15, 6))
plt.title(f'Previsão (LightGBM) vs. Real (RMSE: {lgb_rmse:.2f})')
plt.plot(y_test.index, y_test, label='Valores Reais', alpha=0.7)
plt.plot(y_test.index, lgb_predicoes, label='Previsões (LGBM)', linestyle='--')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd

# --- 14. PLACAR DE MODELOS ---

# 1. Tenta criar um dicionário com os resultados
#    (Isso assume que as células 11, 12 e 13 já rodaram
#    e as variáveis 'rmse', 'rf_rmse', e 'lgb_rmse' existem)

try:
    resultados = {
        'Modelo': [
            'Regressão Linear',
            'Random Forest',
            'LightGBM'
        ],
        'RMSE (Erro Médio)': [
            rmse,
            rf_rmse,
            lgb_rmse
        ]
    }

    # 2. Criar o DataFrame do Placar
    placar_df = pd.DataFrame(resultados)

    # 3. Ordenar do melhor (menor erro) para o pior
    placar_df = placar_df.sort_values(by='RMSE (Erro Médio)', ascending=True)

    # 4. Resetar o índice para uma visualização limpa
    placar_df = placar_df.reset_index(drop=True)

    print("--- 🏆 Placar de Performance dos Modelos ---")
    print("\nO menor RMSE indica o modelo mais preciso:\n")
    print(placar_df)

except NameError:
    print("ERRO: Parece que uma das variáveis de RMSE (rmse, rf_rmse, lgb_rmse) não foi definida.")
    print("Por favor, rode as células 11, 12 e 13 primeiro.")

## Tentar agora fazer a analise para as 10 mais corridas

In [ ]:
import pandas as pd
import numpy as np
import holidays
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb # Importa o lgb
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# --- 1. DEFINIR QUANTAS CORRIDAS VAMOS ANALISAR ---
# Vamos analisar as 10 mais populares. Mude este número se quiser.
N_TOP_CORRIDAS = 10

# Pega os IDs das N corridas mais populares
top_n_ids = corridas_populares.head(N_TOP_CORRIDAS).index.tolist()

# --- 2. CRIAR O CALENDÁRIO DE FERIADOS (UMA VEZ SÓ) ---
# Pegamos todos os anos do dataset principal para criar um único calendário
anos_globais = df['data_viagem'].dt.year.unique()
feriados_br_rr = holidays.BR(state='RR', years=anos_globais)
print(f"Calendário de feriados criado para os anos: {anos_globais}")

# --- 3. INICIAR O PLACAR GERAL ---
# Esta lista vai guardar o resultado de cada corrida
placar_geral = []

print(f"\nPronto para analisar as {N_TOP_CORRIDAS} corridas mais populares.")
print(f"IDs a serem analisados: {top_n_ids}")

In [ ]:
# --- Bloco 2: Nova Célula 7 (O Loop Principal - VERSÃO ROBUSSTA) ---

placar_geral = [] # Zera o placar para garantir (caso rode a célula de novo)

# Passa por cada ID que definimos na célula anterior
for id_alvo in top_n_ids:
    print("-" * 70)
    print(f"--- 🔄 ANALISANDO CORRIDA ID: {id_alvo} ---")

    try:
        # --- 1. FILTRAGEM E AGREGAÇÃO ---
        df_filtrado = df[df['corrida_id'] == id_alvo].copy()
        df_diario = df_filtrado.groupby('data_viagem').size().reset_index(name='total_passagens_dia')
        df_diario = df_diario.set_index('data_viagem')
        df_diario = df_diario.asfreq('D')
        df_diario['total_passagens_dia'] = df_diario['total_passagens_dia'].fillna(0)
        df_diario = df_diario.reset_index()

        # Pular corridas com histórico muito curto
        if len(df_diario) < 100: # Mínimo de ~3 meses de dados
            print(f"Corrida {id_alvo} pulada: histórico muito curto ({len(df_diario)} dias).")
            continue

        # --- 2. FEATURE ENGINEERING (BÁSICA) ---
        df_diario['dia_da_semana'] = df_diario['data_viagem'].dt.dayofweek
        df_diario['dia_do_ano'] = df_diario['data_viagem'].dt.dayofyear
        periodo_ano = 365.25
        periodo_semana = 7
        df_diario['dia_ano_sin'] = np.sin(2 * np.pi * df_diario['dia_do_ano'] / periodo_ano)
        df_diario['dia_ano_cos'] = np.cos(2 * np.pi * df_diario['dia_do_ano'] / periodo_ano)
        df_diario['dia_semana_sin'] = np.sin(2 * np.pi * df_diario['dia_da_semana'] / periodo_semana)
        df_diario['dia_semana_cos'] = np.cos(2 * np.pi * df_diario['dia_da_semana'] / periodo_semana)
        df_diario['eh_feriado'] = df_diario['data_viagem'].isin(feriados_br_rr).astype(int)
        df_diario = df_diario.sort_values('data_viagem')

        # --- 3. FEATURE ENGINEERING (LAGS DINÂMICOS) ---
        # Esta é a correção: só adicionamos lags se os dados existirem.

        # Lista de features base
        features = ['dia_da_semana', 'eh_feriado', 'dia_ano_sin', 'dia_ano_cos', 'dia_semana_sin', 'dia_semana_cos']

        # Adiciona lag_1 (sempre)
        df_diario['lag_1_dia'] = df_diario['total_passagens_dia'].shift(1)
        features.append('lag_1_dia')

        # Adiciona lag_7 (sempre)
        df_diario['lag_7_dias'] = df_diario['total_passagens_dia'].shift(7)
        features.append('lag_7_dias')

        # SÓ ADICIONA LAG_365 SE TIVER MAIS DE 1 ANO DE DADOS
        if len(df_diario) > 365:
            df_diario['lag_365_dias'] = df_diario['total_passagens_dia'].shift(365)
            features.append('lag_365_dias')
            print(f"Status: Corrida {id_alvo} tem > 1 ano. Usando lag_365_dias.")
        else:
            print(f"Status: Corrida {id_alvo} tem < 1 ano. Ignorando lag_365_dias.")

        # --- 4. PREPARAÇÃO DO MODELO ---
        # O dropna() agora só vai remover os 7 primeiros dias (ou 365 se o lag existir)
        df_modelo = df_diario.dropna()

        if len(df_modelo) < 50: # Check de segurança
            print(f"Corrida {id_alvo} pulada: insuficientes dados após lags.")
            continue

        target = 'total_passagens_dia'
        X = df_modelo[features]
        y = df_modelo[target]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        # --- 5. TREINAR E AVALIAR MODELOS ---

        # 1. Regressão Linear
        modelo_lr = LinearRegression()
        modelo_lr.fit(X_train, y_train)
        pred_lr = np.round(modelo_lr.predict(X_test).clip(0))
        rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))

        # 2. Random Forest
        modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        modelo_rf.fit(X_train, y_train)
        pred_rf = np.round(modelo_rf.predict(X_test).clip(0))
        rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))

        # 3. LightGBM
        modelo_lgb = lgb.LGBMRegressor(random_state=42, n_jobs=-1)
        modelo_lgb.fit(X_train, y_train)
        pred_lgb = np.round(modelo_lgb.predict(X_test).clip(0))
        rmse_lgb = np.sqrt(mean_squared_error(y_test, pred_lgb))

        print(f"Corrida {id_alvo} analisada. RMSE (LGBM): {rmse_lgb:.2f}")

        # --- 6. SALVAR RESULTADOS (DENTRO DO TRY) ---
        # Garantir que o append só acontece se tudo acima der certo.
        placar_geral.append({
            'corrida_id': id_alvo,
            'rmse_reg_linear': rmse_lr,
            'rmse_random_forest': rmse_rf,
            'rmse_lightgbm': rmse_lgb,
            'dias_de_dados': len(df_modelo),
            'usou_lag_365': 'lag_365_dias' in features # Informa se o lag foi usado
        })

    except Exception as e:
        print(f"ERRO GERAL ao processar Corrida {id_alvo}: {e}")
        continue # Pula para a próxima corrida

print("-" * 70)
print("✅ Loop de análise concluído.")

In [ ]:
# --- PLACAR GERAL FINAL ---

if not placar_geral:
    print("Nenhuma corrida foi analisada com sucesso.")
else:
    # Converte a lista de resultados em um DataFrame
    placar_geral_df = pd.DataFrame(placar_geral)

    # Ordena pelo melhor modelo (LGBM) do menor erro para o maior
    placar_geral_df = placar_geral_df.sort_values(by='rmse_lightgbm', ascending=True)

    print("--- 🏆 Placar Geral de Performance (Top Corridas) ---")
    print(placar_geral_df)

In [ ]:
# --- GRÁFICO COMPARATIVO DE PERFORMANCE (RMSE) ---
import matplotlib.pyplot as plt
import seaborn as sns

# Verifica se o placar_geral_df existe
if 'placar_geral_df' not in locals() or placar_geral_df.empty:
    print("⚠️ O DataFrame 'placar_geral_df' não foi encontrado ou está vazio.")
    print("Por favor, execute as células anteriores (Célula 7 e 8) primeiro.")
else:
    # 1. Preparar os dados para o gráfico (Formato Longo para o Seaborn)
    df_plot = placar_geral_df.melt(
        id_vars=['corrida_id'],
        value_vars=['rmse_reg_linear', 'rmse_random_forest', 'rmse_lightgbm'],
        var_name='Modelo',
        value_name='RMSE'
    )

    # Melhorar os nomes para a legenda
    df_plot['Modelo'] = df_plot['Modelo'].replace({
        'rmse_reg_linear': 'Regressão Linear',
        'rmse_random_forest': 'Random Forest',
        'rmse_lightgbm': 'LightGBM'
    })

    # 2. Configurar o Gráfico
    plt.figure(figsize=(14, 7))

    # Criar o gráfico de barras agrupadas
    grafico = sns.barplot(
        data=df_plot,
        x='corrida_id',
        y='RMSE',
        hue='Modelo',
        palette='viridis' # Cores bonitas e profissionais
    )

    # 3. Estilização
    plt.title('Comparação de Erro (RMSE) por Modelo e Corrida', fontsize=16)
    plt.xlabel('ID da Corrida', fontsize=12)
    plt.ylabel('RMSE (Menor é melhor)', fontsize=12)
    plt.legend(title='Algoritmo')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    # Adicionar valores em cima das barras (opcional, mas útil)
    for container in grafico.containers:
        grafico.bar_label(container, fmt='%.1f', padding=3, fontsize=9)

    plt.tight_layout()
    plt.show()

    # 4. Análise Rápida Textual
    melhor_modelo_contagem = placar_geral_df[['rmse_reg_linear', 'rmse_random_forest', 'rmse_lightgbm']].idxmin(axis=1).value_counts()
    print("\n--- 🏆 Contagem de Vitórias (Menor Erro) ---")
    print(melhor_modelo_contagem)

In [ ]:
# --- CÉLULA DE PREPARAÇÃO PARA O DASHBOARD ---
# Precisamos rodar o loop novamente para SALVAR os modelos e históricos na memória

import lightgbm as lgb
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

print("--- 💾 Treinando e Salvando Modelos para o Dashboard ---")

# Dicionários para guardar os "cérebros" da IA
banco_de_modelos = {}      # Vai guardar o modelo LightGBM de cada corrida
banco_de_historico = {}    # Vai guardar os dados passados (para calcular lags)
metricas_finais = []

# Loop simplificado (focado em salvar)
for id_alvo in top_n_ids:
    try:
        # 1. Preparação dos Dados (Mesma lógica anterior)
        df_filtrado = df[df['corrida_id'] == id_alvo].copy()
        df_diario = df_filtrado.groupby('data_viagem').size().reset_index(name='total_passagens_dia')
        df_diario = df_diario.set_index('data_viagem')
        df_diario = df_diario.asfreq('D')
        df_diario['total_passagens_dia'] = df_diario['total_passagens_dia'].fillna(0)
        df_diario = df_diario.reset_index()

        if len(df_diario) < 100: continue

        # Feature Engineering
        df_diario['dia_da_semana'] = df_diario['data_viagem'].dt.dayofweek
        df_diario['dia_do_ano'] = df_diario['data_viagem'].dt.dayofyear
        periodo_ano = 365.25
        periodo_semana = 7
        df_diario['dia_ano_sin'] = np.sin(2 * np.pi * df_diario['dia_do_ano'] / periodo_ano)
        df_diario['dia_ano_cos'] = np.cos(2 * np.pi * df_diario['dia_do_ano'] / periodo_ano)
        df_diario['dia_semana_sin'] = np.sin(2 * np.pi * df_diario['dia_da_semana'] / periodo_semana)
        df_diario['dia_semana_cos'] = np.cos(2 * np.pi * df_diario['dia_da_semana'] / periodo_semana)
        df_diario['eh_feriado'] = df_diario['data_viagem'].isin(feriados_br_rr).astype(int)

        # Lags
        df_diario = df_diario.sort_values('data_viagem')
        features = ['dia_da_semana', 'eh_feriado', 'dia_ano_sin', 'dia_ano_cos', 'dia_semana_sin', 'dia_semana_cos']
        df_diario['lag_1_dia'] = df_diario['total_passagens_dia'].shift(1)
        features.append('lag_1_dia')
        df_diario['lag_7_dias'] = df_diario['total_passagens_dia'].shift(7)
        features.append('lag_7_dias')

        tem_lag_365 = False
        if len(df_diario) > 365:
            df_diario['lag_365_dias'] = df_diario['total_passagens_dia'].shift(365)
            features.append('lag_365_dias')
            tem_lag_365 = True

        df_modelo = df_diario.dropna()
        if len(df_modelo) < 50: continue

        # Treino
        target = 'total_passagens_dia'
        X = df_modelo[features]
        y = df_modelo[target]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        modelo_lgb = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
        modelo_lgb.fit(X_train, y_train)

        # --- O PULO DO GATO: SALVAR NO DICIONÁRIO ---
        banco_de_modelos[id_alvo] = {
            'modelo': modelo_lgb,
            'features': features,
            'tem_lag_365': tem_lag_365
        }
        # Salvamos o df_diario inteiro para poder buscar o valor de vendas passadas
        banco_de_historico[id_alvo] = df_diario.set_index('data_viagem')

        print(f"✅ Modelo da Corrida {id_alvo} salvo na memória.")

    except Exception as e:
        print(f"Erro na corrida {id_alvo}: {e}")

print("-" * 30)
print(f"Total de modelos prontos para o Dashboard: {len(banco_de_modelos)}")

In [ ]:
# --- FUNÇÃO DE PREVISÃO COMPLETA (V5 - COM COLUNAS TÉCNICAS) ---

def prever_para_data(data_alvo_str):
    """
    Gera previsões e retorna colunas técnicas (Seno/Cosseno) para análise.
    """
    try:
        data_alvo = pd.to_datetime(data_alvo_str)
    except:
        return pd.DataFrame({"Erro": ["Data inválida"]})

    resultados_previsao = []

    for corrida_id, dados_modelo in banco_de_modelos.items():
        modelo = dados_modelo['modelo']
        feature_names = dados_modelo['features']
        historico = banco_de_historico[corrida_id]

        # --- 1. VERIFICAÇÃO DE SEGURANÇA ---
        ultima_data_real = historico.index.max()
        dias_de_distancia = (data_alvo - ultima_data_real).days
        status_dados = "Ok"
        if dias_de_distancia > 0:
             if dias_de_distancia > 1: status_dados = "⚠️ Lag Ontem Faltando"
             if dias_de_distancia > 7: status_dados = "⛔ Sem Histórico Recente"

        # --- 2. CÁLCULO DAS FEATURES TÉCNICAS ---
        dia_semana = data_alvo.dayofweek
        dia_nome = data_alvo.day_name()
        dia_ano = data_alvo.dayofyear
        periodo_ano = 365.25
        periodo_semana = 7
        eh_feriado = int(data_alvo in feriados_br_rr)

        # Cálculos Matemáticos (Seno/Cosseno)
        val_dia_ano_sin = np.sin(2 * np.pi * dia_ano / periodo_ano)
        val_dia_ano_cos = np.cos(2 * np.pi * dia_ano / periodo_ano)
        val_dia_semana_sin = np.sin(2 * np.pi * dia_semana / periodo_semana)
        val_dia_semana_cos = np.cos(2 * np.pi * dia_semana / periodo_semana)

        # --- 3. BUSCAR HISTÓRICO (LAGS) ---
        try:
            data_lag_1 = data_alvo - pd.Timedelta(days=1)
            val_lag_1 = historico.loc[data_lag_1, 'total_passagens_dia'] if data_lag_1 in historico.index else 0

            data_lag_7 = data_alvo - pd.Timedelta(days=7)
            val_lag_7 = historico.loc[data_lag_7, 'total_passagens_dia'] if data_lag_7 in historico.index else 0

            data_lag_30 = data_alvo - pd.Timedelta(days=30)
            val_lag_30 = historico.loc[data_lag_30, 'total_passagens_dia'] if data_lag_30 in historico.index else 0

            val_lag_365 = 0
            data_lag_365 = data_alvo - pd.Timedelta(days=365)
            if data_lag_365 in historico.index:
                val_lag_365 = historico.loc[data_lag_365, 'total_passagens_dia']
        except:
            val_lag_1, val_lag_7, val_lag_30, val_lag_365 = 0, 0, 0, 0

        # --- 4. PREPARAR INPUT ---
        input_data = {
            'dia_da_semana': [dia_semana],
            'eh_feriado': [eh_feriado],
            'dia_ano_sin': [val_dia_ano_sin],
            'dia_ano_cos': [val_dia_ano_cos],
            'dia_semana_sin': [val_dia_semana_sin],
            'dia_semana_cos': [val_dia_semana_cos],
            'lag_1_dia': [val_lag_1],
            'lag_7_dias': [val_lag_7]
        }
        if dados_modelo['tem_lag_365']:
            input_data['lag_365_dias'] = [val_lag_365]

        X_input = pd.DataFrame(input_data)
        X_input = X_input[[col for col in feature_names if col in X_input.columns]]

        # --- 5. PREVER ---
        if X_input.shape[1] == len(feature_names):
             predicao = modelo.predict(X_input)[0]
             predicao = max(0, round(predicao))
        else:
             predicao = 0

        # --- 6. RETORNO COMPLETO ---
        resultados_previsao.append({
            'Corrida ID': corrida_id,
            'Data': data_alvo.strftime('%d/%m/%Y'),
            'Dia Semana': dia_nome,
            'Status': status_dados,
            'Previsão': int(predicao),
            'Ontem': int(val_lag_1),
            'Semana': int(val_lag_7),
            'Mês': int(val_lag_30),
            'Ano': int(val_lag_365),
            'Feriado': 'Sim' if eh_feriado else 'Não',
            # Colunas Técnicas Solicitadas
            'dia_da_semana': int(dia_semana),
            'dia_ano_sin': float(val_dia_ano_sin),
            'dia_ano_cos': float(val_dia_ano_cos),
            'dia_semana_sin': float(val_dia_semana_sin),
            'dia_semana_cos': float(val_dia_semana_cos)
        })

    return pd.DataFrame(resultados_previsao).sort_values('Previsão', ascending=False)

In [ ]:
# --- DIAGNÓSTICO 1: OLHAR O HISTÓRICO REAL NA MEMÓRIA ---
import pandas as pd

def raio_x_historico(corrida_id, data_alvo_str):
    print(f"--- 🕵️‍♂️ Investigando Corrida {corrida_id} para a data {data_alvo_str} ---")

    if corrida_id not in banco_de_historico:
        print("ERRO: ID não encontrado no banco de histórico.")
        return

    historico = banco_de_historico[corrida_id]
    data_alvo = pd.to_datetime(data_alvo_str)

    # Vamos olhar os 7 dias ANTES da data alvo
    data_inicio = data_alvo - pd.Timedelta(days=7)

    # Filtra o período
    mask = (historico.index >= data_inicio) & (historico.index < data_alvo)
    recorte = historico.loc[mask]

    if recorte.empty:
        print("⚠️ ALERTA: Nenhum dado encontrado nos 7 dias anteriores!")
    else:
        print("\nHistórico da última semana (O que o modelo deveria ver):")
        display(recorte[['total_passagens_dia']])

    # Checagem específica dos pontos de LAG
    print("\n--- Checagem Pontual dos Lags ---")
    for dias in [1, 7, 30, 365]:
        data_lag = data_alvo - pd.Timedelta(days=dias)
        if data_lag in historico.index:
            venda = historico.loc[data_lag, 'total_passagens_dia']
            print(f"Lag {dias} dias ({data_lag.date()}): {venda} vendas")
        else:
            print(f"Lag {dias} dias ({data_lag.date()}): SEM DADOS (Assumido 0)")

# --- EXECUÇÃO DO TESTE ---
# Substitua pela data que você estava testando e o ID problemático
# Exemplo: ID 104 na data '2025-04-12' (conforme seu print)
raio_x_historico(104, '2025-04-12')

In [ ]:
# --- DIAGNÓSTICO 2: O QUE O MODELO ESTÁ PENSANDO? ---
import matplotlib.pyplot as plt
import pandas as pd
import lightgbm as lgb

def ver_cerebro_modelo(corrida_id):
    if corrida_id not in banco_de_modelos:
        print("ID não encontrado.")
        return

    dados = banco_de_modelos[corrida_id]
    modelo = dados['modelo']
    features = dados['features']

    # Extrai a importância
    importancia = modelo.feature_importances_

    # Cria DataFrame para visualizar
    df_imp = pd.DataFrame({
        'Feature': features,
        'Importancia': importancia
    }).sort_values('Importancia', ascending=False)

    print(f"--- Importância das Features para Corrida {corrida_id} ---")
    print("Se 'dia_semana' ou 'dia_ano' estiverem no topo e 'lag' em baixo,")
    print("o modelo está ignorando o histórico recente e 'chutando' pela média histórica.")

    # Plot
    plt.figure(figsize=(10, 6))
    plt.barh(df_imp['Feature'], df_imp['Importancia'], color='teal')
    plt.gca().invert_yaxis()
    plt.title(f'O que define a previsão da Corrida {corrida_id}?')
    plt.xlabel('Peso da Decisão')
    plt.show()

    display(df_imp)

# --- EXECUÇÃO ---
ver_cerebro_modelo(104)

In [ ]:
# --- DIAGNÓSTICO 3: O INPUT EXATO ---
import pandas as pd
import numpy as np

def debug_input_modelo(corrida_id, data_alvo_str):
    print(f"--- Simulando Input para Corrida {corrida_id} em {data_alvo_str} ---")

    # Copia da lógica interna da função de previsão para ver os valores intermediários
    dados_modelo = banco_de_modelos[corrida_id]
    historico = banco_de_historico[corrida_id]
    feature_names = dados_modelo['features']

    data_alvo = pd.to_datetime(data_alvo_str)

    # Recalcula tudo e PRINTA
    data_lag_1 = data_alvo - pd.Timedelta(days=1)
    val_lag_1 = historico.loc[data_lag_1, 'total_passagens_dia'] if data_lag_1 in historico.index else 0
    print(f"Valor LAG 1 (Ontem) calculado: {val_lag_1}")

    data_lag_7 = data_alvo - pd.Timedelta(days=7)
    val_lag_7 = historico.loc[data_lag_7, 'total_passagens_dia'] if data_lag_7 in historico.index else 0
    print(f"Valor LAG 7 (Semana) calculado: {val_lag_7}")

    # Input Final
    input_simulado = pd.DataFrame([{
        'dia_da_semana': data_alvo.dayofweek,
        'eh_feriado': int(data_alvo in feriados_br_rr),
        'dia_ano_sin': np.sin(2 * np.pi * data_alvo.dayofyear / 365.25),
        'dia_ano_cos': np.cos(2 * np.pi * data_alvo.dayofyear / 365.25),
        'dia_semana_sin': np.sin(2 * np.pi * data_alvo.dayofweek / 7),
        'dia_semana_cos': np.cos(2 * np.pi * data_alvo.dayofweek / 7),
        'lag_1_dia': val_lag_1,
        'lag_7_dias': val_lag_7
    }])

    if dados_modelo['tem_lag_365']:
        data_lag_365 = data_alvo - pd.Timedelta(days=365)
        val_365 = historico.loc[data_lag_365, 'total_passagens_dia'] if data_lag_365 in historico.index else 0
        input_simulado['lag_365_dias'] = val_365
        print(f"Valor LAG 365 (Ano) calculado: {val_365}")

    input_simulado = input_simulado[feature_names]

    print("\n--- O Vetor que entra no Modelo ---")
    display(input_simulado)

    pred = dados_modelo['modelo'].predict(input_simulado)[0]
    print(f"\nPREVISÃO RESULTANTE: {pred}")

# --- EXECUÇÃO ---
debug_input_modelo(104, '2025-04-12')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def gerar_grafico_previsao(df_resultado):
    """
    Gera um gráfico de linhas comparativo com base no dataframe de resultado da previsão.
    """
    # ── 1. Preparar os Dados ───────────────────────────────────────────────────
    # Vamos garantir que estamos pegando apenas as colunas numéricas necessárias
    colunas_necessarias = ['Previsão', 'Ontem', 'Semana', 'Mês', 'Ano']

    # Verifica se as colunas existem (se não, o dashboard não gerou direito)
    for col in colunas_necessarias:
        if col not in df_resultado.columns:
            print(f"Erro: Coluna {col} não encontrada nos dados.")
            return

    # Extrai a lista de corridas e converte para string para o eixo X
    corridas = df_resultado['Corrida ID'].astype(str).tolist()

    # Cria um dicionário com os dados no formato que o matplotlib espera
    dados = {col: df_resultado[col].tolist() for col in colunas_necessarias}

    # Pega a data da primeira linha para colocar no título
    data_grafico = df_resultado['Data'].iloc[0]

    # ── 2. Configurações visuais ───────────────────────────────────────────────
    plt.rcParams.update({
        'font.family':      'serif',
        'font.serif':       ['Times New Roman', 'DejaVu Serif'],
        'font.size':        10,
        'axes.linewidth':   0.8,
        'axes.spines.top':  False,
        'axes.spines.right':False,
        'xtick.direction':  'out',
        'ytick.direction':  'out',
        'xtick.major.size': 4,
        'ytick.major.size': 4,
        'xtick.minor.size': 2,
        'ytick.minor.size': 2,
    })

    # Paleta discreta, adequada para impressão P&B e colorida
    cores = {
        'Previsão': '#1a1a2e',
        'Ontem':    '#e63946',
        'Semana':   '#457b9d',
        'Mês':      '#2a9d8f',
        'Ano':      '#f4a261',
    }

    marcadores = {
        'Previsão': 'D',
        'Ontem':    'o',
        'Semana':   's',
        'Mês':      '^',
        'Ano':      'P',
    }

    estilos = {
        'Previsão': '-',
        'Ontem':    '--',
        'Semana':   '-.',
        'Mês':      ':',
        'Ano':      (0, (5, 2, 1, 2)),
    }

    # ── 3. Figura ───────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8.5, 4.5), dpi=300)

    x = np.arange(len(corridas))

    for serie, valores in dados.items():
        ax.plot(
            x, valores,
            linestyle=estilos[serie],
            marker=marcadores[serie],
            color=cores[serie],
            linewidth=1.4,
            markersize=5.5,
            markerfacecolor='white',
            markeredgewidth=1.4,
            label=serie,
            zorder=3,
        )

    # ── 4. Eixos e grade ───────────────────────────────────────────────────────
    ax.set_xticks(x)
    ax.set_xticklabels([f'#{c}' for c in corridas], fontsize=9)
    ax.set_xlabel('Identificador da Corrida', fontsize=10, labelpad=6)
    ax.set_ylabel('Quantidade de Passageiros', fontsize=10, labelpad=6)

    # Define o limite superior do eixo Y um pouco acima do valor máximo encontrado
    ymax = max([max(v) for v in dados.values()])
    ax.set_ylim(-5, ymax + 15)

    ax.yaxis.set_minor_locator(plt.MultipleLocator(5))
    ax.grid(axis='y', which='major', linewidth=0.5, linestyle='--',
            color='#cccccc', zorder=0)
    ax.grid(axis='y', which='minor', linewidth=0.3, linestyle=':',
            color='#e5e5e5', zorder=0)

    # ── 5. Legenda ─────────────────────────────────────────────────────────────
    legend = ax.legend(
        title='Período',
        title_fontsize=9,
        fontsize=9,
        loc='upper right',
        frameon=True,
        framealpha=0.95,
        edgecolor='#aaaaaa',
        handlelength=2.5,
        columnspacing=1.2,
        ncol=1,
    )
    legend.get_frame().set_linewidth(0.6)

    # ── 6. Título e fonte ──────────────────────────────────────────────────────
    ax.set_title(
        f'Previsão de Demanda e Histórico Comparativo por Corrida\n({data_grafico})',
        fontsize=11, pad=10, fontweight='bold',
    )

    fig.text(
        0.99, 0.005,
        'Fonte: O Autor (2025).',
        ha='right', va='bottom',
        fontsize=7.5, style='italic', color='#555555',
    )

    plt.tight_layout(rect=[0, 0.02, 1, 1])

    # ── 7. Exibição (Sem salvar arquivo) ───────────────────────────────────────
    # Retiramos o plt.savefig() para não poluir o ambiente do Colab com muitos arquivos
    # Mas você pode clicar com o botão direito na imagem gerada e salvar
    plt.show()

In [ ]:
# --- DASHBOARD MODERNO E CLEAN (UX OTIMIZADO COM COMPARADOR DE GRÁFICOS) ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import date

# ── 1. FUNÇÃO GERADORA DO GRÁFICO (AGORA DINÂMICA) ─────────────────────────────
def gerar_grafico_previsao(df_resultado, tipo_grafico='Barras'):
    """
    Gera o gráfico comparativo alternando entre Linhas, Barras ou Pirulito.
    """
    colunas_necessarias = ['Previsão', 'Ontem', 'Semana', 'Mês', 'Ano']
    for col in colunas_necessarias:
        if col not in df_resultado.columns: return

    corridas = df_resultado['Corrida ID'].astype(str).tolist()
    dados = {col: df_resultado[col].tolist() for col in colunas_necessarias}
    data_grafico = df_resultado['Data'].iloc[0]

    plt.rcParams.update({'font.family': 'serif', 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

    # Paleta de cores mantida
    cores = {'Previsão': '#1a1a2e', 'Ontem': '#e63946', 'Semana': '#457b9d', 'Mês': '#2a9d8f', 'Ano': '#f4a261'}
    marcadores = {'Previsão': 'D', 'Ontem': 'o', 'Semana': 's', 'Mês': '^', 'Ano': 'P'}

    fig, ax = plt.subplots(figsize=(8.5, 4.5), dpi=300)
    x = np.arange(len(corridas))

    # --- Lógica para alternar o tipo de gráfico ---
    if tipo_grafico == 'Linhas':
        estilos = {'Previsão': '-', 'Ontem': '--', 'Semana': '-.', 'Mês': ':', 'Ano': (0, (5, 2, 1, 2))}
        for serie, valores in dados.items():
            ax.plot(x, valores, linestyle=estilos[serie], marker=marcadores[serie], color=cores[serie],
                    linewidth=1.4, markersize=5.5, markerfacecolor='white', markeredgewidth=1.4, label=serie, zorder=3)

    elif tipo_grafico == 'Barras':
        largura_barra = 0.15
        offsets = [-2, -1, 0, 1, 2] # Deslocamento para agrupar as 5 barras

        for i, (serie, valores) in enumerate(dados.items()):
            posicao_x = x + (offsets[i] * largura_barra)
            ax.bar(posicao_x, valores, largura_barra, color=cores[serie], label=serie, zorder=3, alpha=0.9)

    elif tipo_grafico == 'Pirulito (Lollipop)':
        largura_barra = 0.15
        offsets = [-2, -1, 0, 1, 2]

        for i, (serie, valores) in enumerate(dados.items()):
            posicao_x = x + (offsets[i] * largura_barra)
            # Desenha a "haste" do pirulito
            ax.vlines(x=posicao_x, ymin=0, ymax=valores, color=cores[serie], linewidth=2, zorder=3)
            # Desenha a "bolinha" no topo
            ax.plot(posicao_x, valores, marker=marcadores[serie], color=cores[serie],
                    markersize=7, linestyle='', label=serie, zorder=4)

    # Configurações comuns dos eixos
    ax.set_xticks(x)

    # Adiciona a tag "Rota " apenas se for apenas 1 corrida para ficar bonito
    if len(corridas) == 1:
        ax.set_xticklabels([f'Rota #{c}' for c in corridas], fontsize=11, fontweight='bold')
    else:
        ax.set_xticklabels([f'#{c}' for c in corridas], fontsize=9)

    ax.set_xlabel('Identificador da Rota Comercial', fontsize=10, labelpad=6)
    ax.set_ylabel('Quantidade de Passageiros', fontsize=10, labelpad=6)

    ymax = max([max(v) for v in dados.values()])
    ax.set_ylim(0, ymax + (ymax * 0.2)) # Dá 20% de respiro no topo
    ax.grid(axis='y', which='major', linewidth=0.5, linestyle='--', color='#cccccc', zorder=0)

    legend = ax.legend(title='Comparativo', fontsize=9, loc='upper right', framealpha=0.95, edgecolor='#aaaaaa')
    ax.set_title(f'Análise de Demanda de Passageiros\n({data_grafico})', fontsize=11, pad=10, fontweight='bold')
    plt.tight_layout(rect=[0, 0.02, 1, 1])
    plt.show()

# ── 2. INTERFACE DO DASHBOARD ──────────────────────────────────────────────────
titulo = widgets.HTML("<h2 style='color: #2c3e50;'>🚌 Sistema de Previsão de Demanda - AMATUR</h2>")
instrucao = widgets.HTML("<p style='color: #7f8c8d; font-size: 14px;'>Ajuste os filtros abaixo para visualizar a estimativa.</p>")

date_picker = widgets.DatePicker(description='Data Viagem:', value=date.today(), disabled=False)

rotas_disponiveis = ['Todas', '50', '70', '71', '104', '120', '121', '123', '152', '153', '601']
rota_dropdown = widgets.Dropdown(description='Rota (ID):', options=rotas_disponiveis, value='Todas')

# NOVO: Seletor de visualização de gráfico
tipos_grafico = ['Barras', 'Pirulito (Lollipop)', 'Linhas']
grafico_dropdown = widgets.Dropdown(description='Visual:', options=tipos_grafico, value='Barras')

botao_prever = widgets.Button(description=' Analisar Demanda', button_style='primary', icon='search')
output_area = widgets.Output()

def on_botao_click(b):
    with output_area:
        clear_output()
        if date_picker.value is None:
            print("⚠️ Selecione uma data.")
            return

        print(f"🔄 Consultando inteligência artificial para {date_picker.value}...")

        data_str = str(date_picker.value)
        df_resultado = prever_para_data(data_str)

        if df_resultado.empty:
            print("Nenhum modelo disponível para esta data.")
        else:
            # Filtro de rota
            if rota_dropdown.value != 'Todas':
                df_resultado = df_resultado[df_resultado['Corrida ID'].astype(str) == rota_dropdown.value]
                if df_resultado.empty:
                    print(f"Nenhuma previsão encontrada para a rota {rota_dropdown.value} nesta data.")
                    return

            cols_ordem = ['Corrida ID', 'Status', 'Previsão', 'Ontem', 'Semana', 'Mês', 'Ano']
            cols_finais = [c for c in cols_ordem if c in df_resultado.columns]
            df_final = df_resultado[cols_finais]

            def color_status(val):
                if '⛔' in val: return 'color: #e74c3c; font-weight: bold'
                if '⚠️' in val: return 'color: #f39c12; font-weight: bold'
                return 'color: #27ae60; font-weight: bold'

            def highlight_previsao(s):
                estilos = []
                for col in s.index:
                    if col == 'Previsão':
                        estilos.append('background-color: #e8f8f5; color: #117a65; font-weight: bold; font-size: 15px;')
                    else:
                        estilos.append('')
                return estilos

            estilo_tabela = [
                {'selector': 'th', 'props': [('background-color', '#34495e'), ('color', 'white'), ('text-align', 'center'), ('font-family', 'sans-serif'), ('padding', '10px')]},
                {'selector': 'td', 'props': [('text-align', 'center'), ('border-bottom', '1px solid #ecf0f1'), ('padding', '8px'), ('font-family', 'sans-serif')]},
                {'selector': 'tr:hover', 'props': [('background-color', '#f9f9f9')]}
            ]

            display(df_final.style
                    .applymap(color_status, subset=['Status'])
                    .apply(highlight_previsao, axis=1)
                    .set_table_styles(estilo_tabela)
                    .hide(axis="index"))

            print("\n")
            # Chama o gerador de gráfico passando o tipo selecionado no dropdown
            gerar_grafico_previsao(df_resultado, tipo_grafico=grafico_dropdown.value)

botao_prever.on_click(on_botao_click)

# Organiza os seletores de forma mais limpa (Linha 1: Data e Rota | Linha 2: Grafico e Botao)
linha_1 = widgets.HBox([date_picker, rota_dropdown])
linha_2 = widgets.HBox([grafico_dropdown, botao_prever])
controles = widgets.VBox([linha_1, linha_2])

dashboard = widgets.VBox([titulo, instrucao, controles, widgets.HTML("<hr style='border: 1px solid #ecf0f1;'>"), output_area])

display(dashboard)